# PharmPilot — Clinical Knowledge Bulk Embedder (free Colab T4)

Embeds **FDA drug-label** sections — drug/food **interactions**, **contraindications**, **warnings & precautions**, **pharmacokinetics** — into vectors for the PharmPilot **Second Brain**, on a free **T4 GPU** (~100–1000× faster than a laptop CPU).

**Run all cells.** It downloads the public openFDA/DailyMed label bulk, dedups by generic drug, embeds on the GPU, and produces `pharmpilot_kb_points.jsonl.gz` (auto-downloads). Then **locally**: `python scripts/load_points.py pharmpilot_kb_points.jsonl.gz` → searchable **offline**.

⚠️ **Do not change the embedding model** — it must match the app's query side (`neuml/pubmedbert-base-embeddings`) or retrieval breaks. Source = public-domain FDA labels (no PHI, no licensing issue).

**Scale:** start with the default small knobs (validates in ~minutes), then raise `MAX_PARTITIONS`/`MAX_DRUGS` for the full catalog. Deterministic IDs → re-runs are resumable, no duplicates.

First: **Runtime → Change runtime type → T4 GPU**.


In [ ]:
!nvidia-smi -L || echo "⚠️  No GPU detected — Runtime → Change runtime type → T4 GPU"
!pip -q install sentence-transformers qdrant-client requests tqdm

In [ ]:
import os, re, gzip, json, uuid, zipfile, io, requests, torch
from tqdm.auto import tqdm

EMBED_MODEL = "neuml/pubmedbert-base-embeddings"   # MUST match the app's query model
EMBED_DIM   = 768
CHUNK_SIZE, CHUNK_OVERLAP = 1200, 200
ID_NS = uuid.UUID("a7c1f0de-1234-4abc-9def-0123456789ab")  # same namespace as the repo

# Clinical domains -> openFDA label fields (add/remove freely)
SECTIONS = [
  {"domain":"interaction",      "header":"DRUG & FOOD INTERACTIONS", "fields":["drug_interactions"]},
  {"domain":"contraindication", "header":"CONTRAINDICATIONS",        "fields":["contraindications"]},
  {"domain":"precaution",       "header":"WARNINGS & PRECAUTIONS",   "fields":["boxed_warning","warnings_and_cautions","warnings_and_precautions","precautions","warnings"]},
  {"domain":"pharmacokinetics", "header":"PHARMACOKINETICS",         "fields":["pharmacokinetics","clinical_pharmacology"]},
]

# --- scale knobs: start small to validate, then raise ---
MAX_PARTITIONS = 1      # openFDA splits labels into ~12 files; None = all (~140k labels)
MAX_DRUGS      = 300    # distinct generics to embed; None = no cap
OUT_FILE       = "pharmpilot_kb_points.jsonl.gz"

In [ ]:
# openFDA publishes the FDA/DailyMed drug-label dataset as downloadable JSON partitions
man = requests.get("https://api.fda.gov/download.json", timeout=60).json()
parts = man["results"]["drug"]["label"]["partitions"]
print(f"{len(parts)} label partitions available; partition keys: {list(parts[0].keys())}")
sel = parts if MAX_PARTITIONS is None else parts[:MAX_PARTITIONS]
print(f"using {len(sel)} partition(s)")

def iter_records(partitions):
    for p in partitions:
        url = p.get("file") or p.get("url")
        print("downloading", url.split("/")[-1], f"(~{p.get('size_mb','?')}MB, {p.get('records','?')} records)")
        raw = requests.get(url, timeout=900).content
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            for nm in zf.namelist():
                if nm.endswith(".json"):
                    data = json.loads(zf.read(nm).decode("utf-8", "replace"))
                    for r in data.get("results", []):
                        yield r

In [ ]:
from sentence_transformers import SentenceTransformer
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("embedding device:", dev)
model = SentenceTransformer(EMBED_MODEL, device=dev)

clean = lambda h: re.sub(r"\s+", " ", re.sub(r"<[^>]+>", "", h or "")).strip()

def generic_of(lbl):
    o = lbl.get("openfda", {}) or {}
    for k in ("generic_name", "substance_name"):
        v = o.get(k) or []
        if v: return str(v[0]).strip().lower()
    return None

def sections_of(lbl):
    out = []
    for s in SECTIONS:
        parts = []
        for f in s["fields"]:
            for v in (lbl.get(f) or []):
                c = clean(v)
                if len(c) > 50: parts.append(c)
        if parts: out.append({**s, "text": "\n".join(parts)})
    return out

def chunk(t):
    sents = re.split(r"(?<=[.!?])\s+", t); ch=[]; cur=[]; n=0
    for s in sents:
        if n + len(s) > CHUNK_SIZE and cur:
            c = " ".join(cur).strip()
            if len(c) > 80: ch.append(c)
            tail = c[-CHUNK_OVERLAP:]; cur=[tail]; n=len(tail)
        cur.append(s); n += len(s)
    c = " ".join(cur).strip()
    if len(c) > 80: ch.append(c)
    return ch

seen=set(); n_drugs=n_vec=0
with gzip.open(OUT_FILE, "wt", encoding="utf-8") as out:
    pbar = tqdm(iter_records(sel), desc="labels")
    for lbl in pbar:
        gen = generic_of(lbl)
        if not gen or gen in seen: continue
        secs = sections_of(lbl)
        if not secs: continue
        seen.add(gen); brand = (lbl.get("openfda",{}).get("brand_name") or [gen])[0]
        texts, metas = [], []
        for sec in secs:
            for i, c in enumerate(chunk(sec["text"])):
                texts.append(c); metas.append((sec, i))
        if not texts: continue
        vecs = model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        for (c, (sec, i)), v in zip(zip(texts, metas), vecs):
            pid = str(uuid.uuid5(ID_NS, f"spl:{gen}:{sec['domain']}:{i}"))
            out.write(json.dumps({"id": pid, "vector": v.tolist(), "payload": {
                "content": c, "drug": gen, "domain": sec["domain"],
                "source_title": f"{brand} — {sec['header']}", "source_id": f"spl_{gen}_{sec['domain']}",
                "source_type": "fda_spl", "url": "https://dailymed.nlm.nih.gov/", "chunk_index": i}}) + "\n")
            n_vec += 1
        n_drugs += 1; pbar.set_postfix(drugs=n_drugs, vectors=n_vec)
        if MAX_DRUGS and n_drugs >= MAX_DRUGS: break
print(f"DONE: {n_drugs} distinct drugs, {n_vec} vectors -> {OUT_FILE}")

In [ ]:
sz = os.path.getsize(OUT_FILE) / 1e6
print(f"{OUT_FILE}: {sz:.1f} MB  (download starting…)")
from google.colab import files
files.download(OUT_FILE)

## Make it offline on your machine

1. Move `pharmpilot_kb_points.jsonl.gz` into your PharmPilot repo folder.
2. Start Qdrant: `.qdrant/qdrant --config-path .qdrant/config.yaml`
3. Load (no re-embedding — fast upsert):
   ```
   python scripts/load_points.py pharmpilot_kb_points.jsonl.gz
   ```
The **Second Brain (Alt+R)** now searches the whole set, fully **offline**.

**Bigger corpus:** raise `MAX_PARTITIONS` / `MAX_DRUGS` and re-run. Deterministic IDs mean re-runs (and resumed sessions) refresh in place — no duplicates.
